In [3]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("D:/loan-default-prediction/data/processed/modeling_datset.csv")

model_df = pd.read_csv(DATA_PATH)

print(model_df.shape)
print(model_df["issue_year"].min(), model_df["issue_year"].max())
display(model_df.head())

(1348099, 13)
2007 2018


,loan_amnt,term_months,int_rate,grade,sub_grade,emp_length_years,home_ownership,annual_inc,verification_status,issue_year,purpose,dti,default_flag
0,3600.0,36,13.99,C,C4,10.0,MORTGAGE,55000.0,Not Verified,2015,debt_consolidation,5.91,0
1,24700.0,36,11.99,C,C1,10.0,MORTGAGE,65000.0,Not Verified,2015,small_business,16.06,0
2,20000.0,60,10.78,B,B4,10.0,MORTGAGE,63000.0,Not Verified,2015,home_improvement,10.78,0
3,10400.0,60,22.45,F,F1,3.0,MORTGAGE,104433.0,Source Verified,2015,major_purchase,25.37,0
4,11950.0,36,13.44,C,C3,4.0,RENT,34000.0,Source Verified,2015,debt_consolidation,10.20,0


In [23]:
train_df = model_df[model_df["issue_year"] <= 2015].copy()

validation_df = model_df[
    model_df["issue_year"].between(2016,2017)
].copy()

test_df = model_df[
    model_df["issue_year"] == 2018
].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain years:", train_df["issue_year"].min(), "-", train_df["issue_year"].max())
print("Validaton years:", validation_df["issue_year"].min(), "-", validation_df["issue_year"].max())
print("Test years:", test_df["issue_year"].sort_index().min(), "-", train_df["issue_year"].sort_index().max())

Train shape: (829355, 13)
Validation shape: (462426, 13)
Test shape: (56318, 13)

Train years: 2007 - 2015
Validaton years: 2016 - 2017
Test years: 2018 - 2015


In [24]:
print("Unique test years:")
print(sorted(test_df["issue_year"].unique()))

print("\nTest-year counts:")
print(test_df["issue_year"].value_counts().sort_index())


Unique test years:
[np.int64(2018)]

Test-year counts:
issue_year
2018    56318
Name: count, dtype: int64


In [19]:
print("Correct test range:")
print(test_df["issue_year"].min(), "-", test_df["issue_year"].max())

Correct test range:
2018 - 2018


In [26]:
for name, split in {
    "Train": train_df,
    "Validation": validation_df,
    "Test": test_df,
}.items():
    print(f"{name}:")
    print(" Rows:", len(split))
    print(" Bad-loan rate:", round(split["default_flag"].mean() * 100, 2), "%")
    print()

assert len(train_df) + len(validation_df) + len(test_df) == len(model_df)

assert set(train_df.index).isdisjoint(validation_df.index)
assert set(train_df.index).isdisjoint(test_df.index)
assert set(validation_df.index).isdisjoint(test_df.index)

print("Split size, target-rate, and overlap checks passed.")

Train:
 Rows: 829355
 Bad-loan rate: 18.46 %

Validation:
 Rows: 462426
 Bad-loan rate: 23.23 %

Test:
 Rows: 56318
 Bad-loan rate: 15.76 %

Split size, target-rate, and overlap checks passed.


In [27]:
target_column  = "default_flag"

feature_columns = [
    column for column in model_df.columns
    if column != target_column
]

X_train = train_df[feature_columns]
Y_train = train_df[target_column]

X_validation = validation_df[feature_columns]
Y_validation = validation_df[target_column]

X_test = test_df[feature_columns]
Y_test = test_df[target_column]

print("Feature columns:")
print(feature_columns)

print("\nshapes:")
print("X_train", X_train.shape)
print("Y_train", Y_train.shape)
print("X_validation", X_validation.shape)
print("Y_validation", Y_validation.shape)
print("X_test", X_test.shape)
print("Y_test", Y_test.shape)

Feature columns:
['loan_amnt', 'term_months', 'int_rate', 'grade', 'sub_grade', 'emp_length_years', 'home_ownership', 'annual_inc', 'verification_status', 'issue_year', 'purpose', 'dti']

shapes:
X_train (829355, 12)
Y_train (829355,)
X_validation (462426, 12)
Y_validation (462426,)
X_test (56318, 12)
Y_test (56318,)


In [30]:
assert target_column not in feature_columns
assert X_train.shape[0] == Y_train.shape[0]
assert X_validation.shape[0] == Y_validation.shape[0]
assert X_test.shape[0] == Y_test.shape[0]

print("Feature-target separation passed.")

Feature-target separation passed.


## Time-Based Split Summary

The modeling dataset was split chronologically:

- Training data: 2007–2015
- Validation data: 2016–2017
- Test data: 2018

A chronological split was used instead of a random split to prevent future information from influencing earlier training data. The three sets have no overlapping rows, and their row counts add up to the complete modeling dataset.

The target rate was checked separately in each split to identify possible changes in bad-loan prevalence over time.